# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlaaSherif-Ibrahim/FLYRANK_AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Filled in for **Lane 2 — Refresh / Content Opportunity Scoring**. Sections are in order; each explanation is written by me, and each code cell backs it with real numbers from the starter dataset.


In [1]:
# Setup — get the repo and data, wherever this kernel starts.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AlaaSherif-Ibrahim/FLYRANK_AI"
REPO_DIR = "FLYRANK_AI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found - are you at the repo root?"
print("Starter data found. Ready.")


Working dir: C:\Users\alaa\FLYRANK_AI
Starter data found. Ready.


## 1. My lane as an ML task (type)

**Type: Scoring / Ranking.** Lane 2's question is *"which pages should be reviewed first?"* — a "which ones first?" question. Per the task-type map that maps to **ranking / scoring**: each page gets a continuous priority score, and what the reviewer actually uses is the **order** of that score.

Why not plain classification:

- Under the hood there is a per-page probability (is this page in the declining bucket?), but the decision is not "yes/no per page" — it is "which 50 pages do I open first this week."
- The final output of the pipeline is `final_refresh_score` (0-100), and pages are ranked on it to build the review queue.
- So I will model a probability (classification machinery) but **frame and evaluate it as ranking** (precision@K).


In [2]:
# The deliverable is a ranked queue. The shape of the decision:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# A transparent PLACEHOLDER ordering just to show the output shape (a real model score comes later).
df["placeholder_score"] = df["impressions_90d"] * df["is_declining_label"]
queue = df.nlargest(10, "placeholder_score")[["content_id", "impressions_90d", "is_declining_label", "placeholder_score"]]
queue.insert(0, "rank", range(1, len(queue) + 1))

print(f"inventory: {len(df):,} pages | reviewer capacity: ~50/cycle")
print("-> the output is an ordered shortlist, not a per-page yes/no. Top of a placeholder queue:")
queue


inventory: 30,000 pages | reviewer capacity: ~50/cycle
-> the output is an ordered shortlist, not a per-page yes/no. Top of a placeholder queue:


,rank,content_id,impressions_90d,is_declining_label,placeholder_score
6653,1,content_5fe46e04994d,517715,1,517715
26844,2,content_8c19996aa890,509252,1,509252
21819,3,content_4c36c775b818,463103,1,463103
29879,4,content_1a9e894be2e2,416180,1,416180
13537,5,content_2c2606c5d176,347399,1,347399
26531,6,content_cb112fce36be,309910,1,309910
21565,7,content_9532f197bbc8,309192,1,309192
27478,8,content_008fb02c46cb,236803,1,236803
23767,9,content_813e88069237,233561,1,233561
26304,10,content_ff94c9b6b411,228566,1,228566


## 2. Target or proxy

**The target I will predict: `is_declining_label`** — 1 when the page is in the "declining" bucket, else 0.

**Honest classification: it is a proxy, not an observed future outcome.** The label is a **defined rule** computed inside the same 90-day window: `trend_direction` comes from `trend_pct` = (last-30d vs prev-30d impressions), and the label is `trend_direction == "down"`. It stands in for "this page is currently losing visibility" — it does not observe what happens next.

Two consequences I will keep visible all project:

1. **Leakage guard:** because the label derives from `trend_pct`/`trend_direction`, those two columns (and their 30-day comparison inputs) are **never** model features. The model feature list (18 numeric + 8 categorical) excludes them.
2. **Stronger capstone target (later):** a future-window, observed label built on the warehouse daily facts — e.g. *prior 90 days of features -> did impressions decline over the next 30 days?* The starter slice cannot observe that future; it ships only the current-window bucket. This W02 notebook frames the starter proxy; the capstone can upgrade the target.


In [3]:
# The target column and where it comes from.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("label counts (0 = not declining, 1 = declining):")
print(df["is_declining_label"].value_counts().sort_index())
print(f"label rate: {df['is_declining_label'].mean():.1%}")

print("\nthe label is DERIVED from a same-window rule (trend_pct -> trend_direction):")
print(df.groupby("trend_direction")["is_declining_label"].mean().round(3))

# Leakage guard check: the label's source columns must not be features.
import sys
sys.path.insert(0, "scripts")
import ml_utils
model_features = ml_utils.MODEL_NUMERIC_FEATURES + ml_utils.MODEL_CATEGORICAL_FEATURES
print(f"\nmodel feature count: {len(model_features)} ({len(ml_utils.MODEL_NUMERIC_FEATURES)} numeric + "
      f"{len(ml_utils.MODEL_CATEGORICAL_FEATURES)} categorical)")
print("leakage guard holds:", ("trend_direction" not in model_features) and ("trend_pct" not in model_features))


label counts (0 = not declining, 1 = declining):
is_declining_label
0    13738
1    16262
Name: count, dtype: int64
label rate: 54.2%

the label is DERIVED from a same-window rule (trend_pct -> trend_direction):
trend_direction
down      1.0
flat      0.0
new       0.0
stable    0.0
up        0.0
Name: is_declining_label, dtype: float64

model feature count: 26 (18 numeric + 8 categorical)
leakage guard holds: True


## 3. Success metric

**Primary: Precision@50.** Of the top-50 pages the editor reviews first (fixed capacity ~50/cycle), how many actually carry the declining label? That is the number that matches the real decision: the reviewer uses a shortlist, so what matters is how many of those slots are well spent.

**Secondary: Average Precision** — because the whole ranking matters too (the queue is longer than the first review pass).

**Reference points on this slice (shipped run, client holdout):**

- Random picks would score about the base rate: **54.2%**.
- Hand-rule baseline Precision@50 = **0.240** (~12/50).
- Random forest Precision@50 = **0.740** (~37/50).

So "good" is a Precision@50 clearly above both the base rate and the transparent rule — re-earned on the warehouse, not carried over from this slice.


In [4]:
# The one number I will defend: precision at the top of the queue.
base_rate = df["is_declining_label"].mean()
baseline_p50, model_p50 = 0.240, 0.740   # shipped outputs/model_report.md, client holdout

print(f"base rate (what random picks would score):            {base_rate:.1%}")
print(f"hand-rule baseline  Precision@50 = {baseline_p50:.3f} -> {baseline_p50*50:.0f} of top-50 correct")
print(f"random forest model Precision@50 = {model_p50:.3f} -> {model_p50*50:.0f} of top-50 correct")
print("\nmetric chosen to match the decision: a reviewer only uses ~50 slots per cycle.")


base rate (what random picks would score):            54.2%
hand-rule baseline  Precision@50 = 0.240 -> 12 of top-50 correct
random forest model Precision@50 = 0.740 -> 37 of top-50 correct

metric chosen to match the decision: a reviewer only uses ~50 slots per cycle.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (a page).** 30,000 rows = 30,000 pages across 32 pseudonymized clients. `content_id` is unique per row; `client_id` groups pages into clients (used for client-holdout splits, never as a feature).

The dataframe below is the actual lane slice: every page, the observable signals, and the target column added on. There is also a **review-ready slice** (declining + still getting >= 100 impressions/90d) — that is the pool the ranked queue must be right about first.


In [5]:
# The unit of analysis, as an actual dataframe: ONE ROW = ONE PAGE.
cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "sessions_90d",
        "ctr", "avg_position", "content_age_days", "days_since_last_update", "is_declining_label"]
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"rows: {len(df):,}  | unique content_id: {df['content_id'].nunique():,} -> every row is one page")
print(f"clients: {df['client_id'].nunique()}")
print("\nsample rows (page-level, with the target added):")
df[cols].head(5)

ready = df[(df["is_declining_label"] == 1) & (df["impressions_90d"] >= 100)]
print(f"\nreview-ready slice (declining + >=100 impressions/90d): {len(ready):,} pages")


rows: 30,000  | unique content_id: 30,000 -> every row is one page
clients: 32

sample rows (page-level, with the target added):

review-ready slice (declining + >=100 impressions/90d): 13,152 pages


## 5. Why ML beats a fixed rule here

A hand-written if-statement is not enough because the pattern is real but messy:

1. **Too many interacting signals.** The model works with 26 features (18 numeric + 8 categorical) — search volume, competition, CTR, position, age, freshness, engagement, volume tiers. No fixed set of thresholds captures how *position x age x freshness x engagement* combine.
2. **It shifts across clients and over time.** 32 clients, different histories, heavy-tailed traffic — a rule tuned on one part drifts on another. Client-holdout validation exists precisely because of this.
3. **Empirical evidence it is learnable beyond the rule.** On this slice, the transparent hand-rule scores Precision@50 = 0.240; the learned model scores 0.740 — about 12 vs 37 of the top-50 slots right. That gap is the measurable reason ML earns its place.

Caveat, kept honest: this gap is measured on the **starter slice only**. The warehouse run must re-earn it with its own validation — and the rule stays as the baseline to beat, with reason codes keeping the queue explainable for the editor.


In [6]:
# Why a fixed rule is not enough: the signal count + the measured gap.
import sys
sys.path.insert(0, "scripts")
import ml_utils

num, cat = ml_utils.MODEL_NUMERIC_FEATURES, ml_utils.MODEL_CATEGORICAL_FEATURES
print(f"the model weighs {len(num)} numeric + {len(cat)} categorical = {len(num)+len(cat)} signals")
print("\nnumeric signals:", num)
print("\ncategorical signals:", cat)

# Measured gap on the starter slice (shipped run, client holdout):
baseline_p50, model_p50 = 0.240, 0.740
print(f"\nprecision@50: hand-rule baseline {baseline_p50:.3f} vs learned model {model_p50:.3f} "
      f"-> about {baseline_p50*50:.0f} vs {model_p50*50:.0f} of the top-50 slots correct")
print("caveat: starter slice only; the warehouse run must re-earn this.")


the model weighs 18 numeric + 8 categorical = 26 signals

numeric signals: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

categorical signals: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

precision@50: hand-rule baseline 0.240 vs learned model 0.740 -> about 12 vs 37 of the top-50 slots correct
caveat: starter slice only; the warehouse run must re-earn this.


## Self-check

Before submitting, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
